# JOILang A6000 Strong Prompt Compression Smoke Notebook

A6000에서 A100 기준 strong staged prompt-compression 코드가 실제로 동작하는지 확인합니다.

실행 순서: 위에서 아래로 실행하면 됩니다. `RESET_A6000_TO_ORIGIN_MAIN`은 기본 `False`입니다.

In [2]:
from pathlib import Path
import os, sys, re, json, time, getpass, subprocess
from datetime import datetime
import pandas as pd
import numpy as np

SERVER_PRESET = "A6000_SET_A"
REPO = Path("/home/mgjeong/Desktop/llm/JOILang-Server").resolve()
VERSION_DIR = REPO / "gpt_mg/version0_15_update20260413"
SCRIPT = VERSION_DIR / "scripts/run_ga_search.py"
RESULTS_ROOT = VERSION_DIR / "results"
PYTHON = next((p for p in [
    Path("/home/mgjeong/miniconda3/envs/paper-gpu/bin/python"),
    Path("/home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10"),
    Path(sys.executable),
] if p.exists()), Path(sys.executable)).resolve()
LOCAL_MODEL_BASE = Path("/home/mgjeong/Desktop/llm/local_models").resolve()
MODEL_DIRS = {
    "qwen25_coder_7b": "qwen25_coder_7b",
    "llama31_8b": "llama31_8b",
    "qwen25_coder_14b": "qwen25_coder_14b",
    "phi35_mini": "phi35_mini",
    "gemma2_9b_it": "gemma2_9b_it",
}
MODEL_LIST = [("7B", "qwen25_coder_7b"), ("8B", "llama31_8b"), ("14B", "qwen25_coder_14b")]
MAIN_MODEL_KEY = "qwen25_coder_14b"

print("="*120)
print("[A6000 CONFIG]")
for k, v in {"REPO":REPO, "SCRIPT":SCRIPT, "PYTHON":PYTHON, "LOCAL_MODEL_BASE":LOCAL_MODEL_BASE}.items():
    print(f"{k}: {v} exists={Path(v).exists()}")
print("="*120)
assert REPO.exists(), REPO
assert SCRIPT.exists(), SCRIPT
assert PYTHON.exists(), PYTHON
assert LOCAL_MODEL_BASE.exists(), LOCAL_MODEL_BASE

[A6000 CONFIG]
REPO: /home/mgjeong/Desktop/llm/JOILang-Server exists=True
SCRIPT: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py exists=True
PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10 exists=True
LOCAL_MODEL_BASE: /home/mgjeong/Desktop/llm/local_models exists=True


In [3]:
# Optional: A6000을 origin/main으로 맞추기. 로컬 변경사항을 버리므로 필요할 때만 True.
RESET_A6000_TO_ORIGIN_MAIN = False

def run_cmd(cmd, cwd=REPO, check=True):
    print("\nRUN:", " ".join(map(str, cmd)))
    r = subprocess.run(list(map(str, cmd)), cwd=str(cwd), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.stdout:
        print(r.stdout)
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed rc={r.returncode}: {' '.join(map(str, cmd))}")
    return r

print("="*120); print("[GIT STATUS]"); print("="*120)
run_cmd(["git", "status", "-sb"], check=False)
run_cmd(["git", "log", "-1", "--oneline"], check=False)
if RESET_A6000_TO_ORIGIN_MAIN:
    run_cmd(["git", "fetch", "origin"])
    run_cmd(["git", "reset", "--hard", "origin/main"])
    run_cmd(["git", "clean", "-fd"], check=False)
    run_cmd(["git", "log", "-1", "--oneline"])

print("="*120); print("[STRONG COMPRESSION KEYWORD CHECK]"); print("="*120)
all_script_text = "\n".join(p.read_text(encoding="utf-8", errors="replace") for p in sorted((VERSION_DIR/"scripts").glob("*.py")))
required_keywords = [
    "Advisor Case A", "Advisor Case B", "Advisor Case C",
    "block_compression_proposals", "multi_block_compression_proposals", "global_budget_compression_proposals",
    "block_token_breakdown", "prompt_token_breakdown", "advisor_mutation_summary",
    "accepted_proposals", "rejected_proposals", "compression_fallback",
    "new_by_micro_compression", "new_by_block_compression",
]
keyword_df = pd.DataFrame([{"keyword": k, "exists": k in all_script_text} for k in required_keywords])
display(keyword_df)
missing = keyword_df.loc[~keyword_df["exists"], "keyword"].tolist()
if missing:
    raise RuntimeError(f"Strong compression code appears incomplete. Missing keywords: {missing}")
print("[OK] Strong compression keywords are present.")

[GIT STATUS]

RUN: git status -sb
## main...origin/main
?? JOILang_PromptOps_Cloud_A6000-Update_TokenDown.ipynb
?? gpt_mg/version0_15_update20260413/notebooks/JOILang_A6000_Strong_Compression_Smoke.ipynb


RUN: git log -1 --oneline
89ede499 update

[STRONG COMPRESSION KEYWORD CHECK]


,keyword,exists
0,Advisor Case A,True
1,Advisor Case B,True
2,Advisor Case C,True
3,block_compression_proposals,True
4,multi_block_compression_proposals,True
5,global_budget_compression_proposals,True
6,block_token_breakdown,True
7,prompt_token_breakdown,True
8,advisor_mutation_summary,True
9,accepted_proposals,True


[OK] Strong compression keywords are present.


In [4]:
print("="*120); print("[PYTHON / CUDA CHECK]"); print("="*120)
env_check_code = "\n".join([
    "import sys",
    "print('python:', sys.executable)",
    "for m in ['torch','transformers','accelerate','safetensors','huggingface_hub','pandas','numpy']:",
    "    try:",
    "        mod=__import__(m); print(m, 'OK', getattr(mod, '__version__', ''))",
    "    except Exception as e:",
    "        print(m, 'FAIL', repr(e))",
    "import torch",
    "print('cuda available:', torch.cuda.is_available())",
    "print('cuda device count:', torch.cuda.device_count())",
    "[print(f'device{i}:', torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else None",
    "from transformers import AutoModelForCausalLM",
    "import transformers",
    "print('transformers.is_torch_available:', transformers.is_torch_available())",
    "print('AutoModelForCausalLM import OK')",
])
print(subprocess.check_output([str(PYTHON), "-c", env_check_code], text=True, stderr=subprocess.STDOUT))

print("="*120); print("[MODEL PATH CHECK]"); print("="*120)
rows=[]
for model_key, dirname in MODEL_DIRS.items():
    p = LOCAL_MODEL_BASE / dirname
    rows.append({
        "model_key": model_key,
        "path": str(p),
        "realpath": str(p.resolve()) if p.exists() else None,
        "exists": p.exists(),
        "config": (p/"config.json").exists(),
        "tokenizer": (p/"tokenizer.json").exists(),
        "index": (p/"model.safetensors.index.json").exists(),
        "safetensors_count": len(list(p.glob("*.safetensors"))) if p.exists() else 0,
    })
model_df = pd.DataFrame(rows)
display(model_df)
for _, model_key in MODEL_LIST:
    p = LOCAL_MODEL_BASE / MODEL_DIRS[model_key]
    assert p.exists(), f"Missing model dir: {p}"
    assert (p/"config.json").exists(), f"Missing config.json: {p}"
    assert (p/"tokenizer.json").exists(), f"Missing tokenizer.json: {p}"
    assert (p/"model.safetensors.index.json").exists(), f"Missing index: {p}"
    assert len(list(p.glob("*.safetensors"))) > 0, f"No safetensors: {p}"
print("[OK] Required 3 models are available.")

[PYTHON / CUDA CHECK]
python: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10
torch OK 2.9.1+cu128
transformers OK 4.57.3
accelerate OK 1.12.0
safetensors OK 0.7.0
huggingface_hub OK 0.36.0
pandas OK 2.3.3
numpy OK 2.2.6
cuda available: True
cuda device count: 2
device0: NVIDIA RTX A6000
device1: NVIDIA RTX A6000
transformers.is_torch_available: True
AutoModelForCausalLM import OK

[MODEL PATH CHECK]


,model_key,path,realpath,exists,config,tokenizer,index,safetensors_count
0,qwen25_coder_7b,/home/mgjeong/Desktop/llm/local_models/qwen25_...,/home/mgjeong/.cache/huggingface/hub/models--Q...,True,True,True,True,4
1,llama31_8b,/home/mgjeong/Desktop/llm/local_models/llama31_8b,/home/mgjeong/.cache/huggingface/hub/models--m...,True,True,True,True,4
2,qwen25_coder_14b,/home/mgjeong/Desktop/llm/local_models/qwen25_...,/home/mgjeong/.cache/huggingface/hub/models--Q...,True,True,True,True,6
3,phi35_mini,/home/mgjeong/Desktop/llm/local_models/phi35_mini,None,False,False,False,False,0
4,gemma2_9b_it,/home/mgjeong/Desktop/llm/local_models/gemma2_...,None,False,False,False,False,0


[OK] Required 3 models are available.


In [5]:
REQUIRE_OPENAI_API_KEY = True

def ensure_openai_api_key(required=True):
    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if key:
        print("[OK] OPENAI_API_KEY is already set.")
        return True
    if not required:
        print("[WARN] OPENAI_API_KEY is not set. Advisor smoke will be skipped.")
        return False
    print("[INPUT REQUIRED] OPENAI_API_KEY is not set.")
    key = getpass.getpass("OPENAI_API_KEY: ").strip()
    if not key:
        raise RuntimeError("OPENAI_API_KEY was not provided.")
    os.environ["OPENAI_API_KEY"] = key
    print("[OK] OPENAI_API_KEY has been set for this notebook process.")
    return True

OPENAI_KEY_READY = ensure_openai_api_key(REQUIRE_OPENAI_API_KEY)

[OK] OPENAI_API_KEY is already set.


In [6]:
def get_supported_flags():
    try:
        h = subprocess.check_output([str(PYTHON), str(SCRIPT), "--help"], cwd=str(REPO), text=True, stderr=subprocess.STDOUT, timeout=60)
        return set(re.findall(r"(--[a-zA-Z0-9][a-zA-Z0-9_-]*)", h))
    except Exception as e:
        print("[WARN] --help failed:", repr(e)); return set()
SUPPORTED_FLAGS = get_supported_flags()
print("[SUPPORTED FLAGS COUNT]", len(SUPPORTED_FLAGS))

def add_flag(cmd, flag, value=True, *, repeat_values=False):
    if SUPPORTED_FLAGS and flag not in SUPPORTED_FLAGS:
        print(f"[SKIP unsupported flag] {flag}"); return
    if isinstance(value, bool):
        if value: cmd.append(flag)
    elif value is not None:
        if repeat_values:
            for v in value: cmd.extend([flag, str(v)])
        else:
            cmd.extend([flag, str(value)])

def make_env(model_key, debug_log):
    env = os.environ.copy()
    model_dir = (LOCAL_MODEL_BASE / MODEL_DIRS[model_key]).resolve()
    env.update({
        "JOI_V15_WORKER_PYTHON": str(PYTHON),
        "JOI_V15_LOCAL_MODEL_BASE_DIR": str(LOCAL_MODEL_BASE),
        "JOI_V15_LOCAL_MODEL_NAME": str(model_dir),
        "JOI_V15_LOCAL_DEVICE": "cuda:0",
        "JOI_V15_LOCAL_FILES_ONLY": "true",
        "JOI_V15_WORKER_DEBUG_LOG": str(debug_log),
        "TRANSFORMERS_OFFLINE": env.get("TRANSFORMERS_OFFLINE", "1"),
        "HF_HUB_OFFLINE": env.get("HF_HUB_OFFLINE", "1"),
        "TOKENIZERS_PARALLELISM": env.get("TOKENIZERS_PARALLELISM", "false"),
    })
    return env

def run_streaming(cmd, env):
    print("COMMAND:"); print(" ".join(map(str, cmd))); print("="*120)
    proc = subprocess.Popen(list(map(str, cmd)), cwd=str(REPO), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    print("\nRETURN CODE:", rc)
    if rc != 0:
        raise RuntimeError(f"run_ga_search failed with return code {rc}")

def run_ga_all_categories(model_key=MAIN_MODEL_KEY, categories=(1,), limit_per_category=1, sample_size=1, validation_size=1,
                          population=4, gens=3, target_detpass=90, base_prefix=None, use_advisor=False,
                          timeout_sec=1200, advisor_trigger_mode="always", enable_render_budget_compression=False):
    categories = tuple(categories)
    mode = "cloud_advisor" if use_advisor else "cloudless"
    cats = "".join(map(str, categories))
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_prefix = base_prefix or f"smoke_strong_{SERVER_PRESET}"
    out_dir = RESULTS_ROOT / f"{base_prefix}_{mode}_cat{cats}_lpc{limit_per_category}_pop{population}_gens{gens}_{model_key}_{ts}" / "ga_output"
    out_dir.mkdir(parents=True, exist_ok=True)
    debug_log = Path(f"/tmp/joi_v15_worker_debug_{model_key}_{mode}_{ts}.log")
    env = make_env(model_key, debug_log)
    print("\n"+"="*120); print(f"RUN: {model_key} / {mode}"); print("OUTPUT:", out_dir)
    print("PYTHON:", PYTHON); print("MODEL_BASE:", env["JOI_V15_LOCAL_MODEL_BASE_DIR"]); print("MODEL_NAME:", env["JOI_V15_LOCAL_MODEL_NAME"]); print("DEBUG_LOG:", debug_log); print("="*120)
    cmd = [str(PYTHON), "-u", str(SCRIPT)]
    pairs = [
        ("--profile", "version0_15"), ("--model-key", model_key), ("--target-detpass", target_detpass), ("--llm-mode", "worker"),
        ("--population", population), ("--gens", gens), ("--min-generations", gens), ("--max-generations", gens),
        ("--sample-size", sample_size), ("--validation-size", validation_size), ("--cheap-eval-limit", 2), ("--candidate-k", 1),
        ("--repair-attempts", 0), ("--det-profile", "strict"), ("--selection-mode", "redesign"), ("--fitness-mode", "phase_aware"),
        ("--mutation-mode", "cloudless_decompiler"), ("--category-balance-mode", "guard"), ("--token-penalty-mode", "hybrid"),
        ("--stop-controller-mode", "active"), ("--plateau-window", 1), ("--disruptive-max-attempts", 1),
        ("--reasoning-mutation-mode", "auto"), ("--intent-hint-mode", "auto"), ("--progress", "verbose"),
        ("--timeout-sec", timeout_sec), ("--retries", 0), ("--limit-per-category", limit_per_category), ("--output-root", out_dir),
        ("--compression-detpass-threshold", 90), ("--compression-child-quota", 1), ("--compression-child-ratio", 0.25),
        ("--advisor-compression-child-quota", 1), ("--advisor-prefer-compression-after-detpass", 90),
        ("--compression-token-reduction-target", 0.15), ("--compression-token-plateau-delta", 1.0),
        ("--micro-compression-child-quota", 1), ("--micro-compression-child-ratio", 0.25),
        ("--block-compression-child-quota", 1), ("--block-compression-child-ratio", 0.25),
        ("--multi-block-compression-child-quota", 1), ("--multi-block-compression-child-ratio", 0.25),
        ("--global-budget-compression-child-quota", 0), ("--min-compression-token-delta", 50),
    ]
    for f, v in pairs: add_flag(cmd, f, v)
    for f in ["--feedback-guided-mutation", "--enable-compression-mutation", "--enable-prompt-decompiler", "--enable-rendered-prompt-dedupe", "--enable-pareto-archive", "--enable-group-specialist-archives", "--full-run", "--aggressive-compression-after-target", "--allow-aggressive-compression", "--enable-block-token-breakdown", "--enable-multi-block-compression"]:
        add_flag(cmd, f, True)
    add_flag(cmd, "--enable-render-budget-compression", enable_render_budget_compression)
    add_flag(cmd, "--category", categories, repeat_values=True)
    if use_advisor:
        for f, v in [("--llm-mutation-advisor", True), ("--advisor-model-key", "gpt41_mini"), ("--advisor-trigger-mode", advisor_trigger_mode), ("--advisor-min-population-for-child", 4), ("--advisor-force-child-quota", True)]:
            add_flag(cmd, f, v)
    else:
        add_flag(cmd, "--advisor-trigger-mode", "off")
    run_streaming(cmd, env)
    print("OUTPUT:", out_dir)
    return out_dir
print("[OK] run wrapper ready")

[SUPPORTED FLAGS COUNT] 133
[OK] run wrapper ready


In [7]:
def read_json(path):
    path=Path(path)
    if not path.exists(): return None
    return json.loads(path.read_text(encoding="utf-8", errors="replace"))

def read_jsonl(path):
    rows=[]; path=Path(path)
    if not path.exists(): return rows
    for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
        if line.strip():
            try: rows.append(json.loads(line))
            except Exception: rows.append({"_raw": line})
    return rows

def inspect_run(out_dir):
    out_dir=Path(out_dir)
    print("\n"+"#"*120); print("INSPECT RUN:", out_dir); print("#"*120)
    for f in ["ga_summary.json","ga_generation_progress.csv","population_transitions.csv","mutation_proposals.jsonl","advisor_mutation_proposals.jsonl","advisor_mutation_summary.csv","block_token_breakdown.json","prompt_token_breakdown.json"]:
        print(f"{f:38s}", (out_dir/f).exists())
    if (out_dir/"ga_generation_progress.csv").exists():
        print("\n[GA PROGRESS]")
        progress=pd.read_csv(out_dir/"ga_generation_progress.csv")
        cols=["generation","validation_det_pass_rate","validation_avg_det_score","best_so_far_DETPass","avg_prompt_tokens","compression_phase","compression_ready","aggressive_compression"]
        cols=[c for c in cols if c in progress.columns]
        display(progress[cols])
        if "avg_prompt_tokens" in progress.columns and len(progress):
            t=pd.to_numeric(progress["avg_prompt_tokens"], errors="coerce"); print("token_delta:", t.iloc[-1]-t.iloc[0])
    if (out_dir/"population_transitions.csv").exists():
        print("\n[POPULATION TRANSITIONS]")
        trans=pd.read_csv(out_dir/"population_transitions.csv")
        cols=["generation","compression_ready","compression_phase","new_by_compression","new_by_micro_compression","new_by_block_compression","new_by_multi_block_compression","new_by_global_budget_compression","new_by_compression_fallback","new_by_advisor"]
        cols=[c for c in cols if c in trans.columns]
        display(trans[cols])
    print("\n[ADVISOR PROMPT KEYWORDS]")
    for p in sorted(out_dir.glob("advisor_prompt_generation_*.txt")):
        text=p.read_text(encoding="utf-8", errors="replace")
        print(p.name, "chars", len(text))
        for k in ["Advisor Case A","Advisor Case B","Advisor Case C","prompt_token_breakdown","block_token_breakdown","block_compression_proposals","multi_block_compression_proposals","global_budget_compression_proposals"]:
            print(f"  {k:38s}", k in text)
    print("\n[ADVISOR RESPONSES]")
    for p in sorted(out_dir.glob("advisor_response_generation_*.json")):
        obj=read_json(p); parsed=obj.get("parsed",{}) if isinstance(obj,dict) else {}
        print(p.name, "parsed_keys", list(parsed.keys()) if isinstance(parsed,dict) else None, "accepted", len(obj.get("accepted_proposals",[]) or []) if isinstance(obj,dict) else None, "rejected", len(obj.get("rejected_proposals",[]) or []) if isinstance(obj,dict) else None)
    print("\n[ADVISOR MUTATION SUMMARY]")
    sp=out_dir/"advisor_mutation_summary.csv"
    if sp.exists():
        sdf=pd.read_csv(sp); print("rows", len(sdf)); display(sdf)
    print("\n[ADVISOR MUTATION PROPOSALS]")
    rows=read_jsonl(out_dir/"advisor_mutation_proposals.jsonl"); print("rows", len(rows))
    if rows:
        df=pd.DataFrame(rows); cols=["generation","source","proposal_id","schema_source","mutation_family","compression_level","operator","mutation_type","target_block_id","selected_block_id","selected_block_ids","target_block_family","expected_token_delta","measured_prompt_token_delta","accepted","rejection_reason","fallback_reason"]
        cols=[c for c in cols if c in df.columns]; display(df[cols])
    print("\n[ALL COMPRESSION PROPOSALS]")
    rows=[]
    for fn in ["mutation_proposals.jsonl","advisor_mutation_proposals.jsonl"]:
        for obj in read_jsonl(out_dir/fn):
            op=str(obj.get("operator", obj.get("mutation_type", "")))
            if obj.get("mutation_family")=="compression" or any(x in op for x in ["compress","few_shot","token","budget"]):
                obj["_file"]=fn; rows.append(obj)
    print("rows", len(rows))
    if rows:
        df=pd.DataFrame(rows); cols=["_file","generation","source","schema_source","mutation_family","compression_level","operator","mutation_type","target_block_id","selected_block_id","selected_block_ids","expected_token_delta","measured_prompt_token_delta","accepted","rejection_reason","fallback_reason"]
        cols=[c for c in cols if c in df.columns]; display(df[cols])
print("[OK] inspect_run ready")

[OK] inspect_run ready


In [8]:
cloudless_out = run_ga_all_categories(use_advisor=False, base_prefix=f"smoke_cloudless_strong_{SERVER_PRESET}")
inspect_run(cloudless_out)


RUN: qwen25_coder_14b / cloudless
OUTPUT: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/smoke_cloudless_strong_A6000_SET_A_cloudless_cat1_lpc1_pop4_gens3_qwen25_coder_14b_20260611_195742/ga_output
PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10
MODEL_BASE: /home/mgjeong/Desktop/llm/local_models
MODEL_NAME: /home/mgjeong/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-14B-Instruct/snapshots/aedcc2d42b622764e023cf882b6652e646b95671
DEBUG_LOG: /tmp/joi_v15_worker_debug_qwen25_coder_14b_cloudless_20260611_195742.log
COMMAND:
/home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10 -u /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --model-key qwen25_coder_14b --target-detpass 90 --llm-mode worker --population 4 --gens 3 --min-generations 3 --max-generations 3 --sample-size 1 --validation-size 1 --cheap-eval-limit 2 --candidate-k 1 --repair-attempts 0 --det-profile

,generation,validation_det_pass_rate,validation_avg_det_score,best_so_far_DETPass,avg_prompt_tokens,compression_phase,compression_ready
0,1,100.0,100.0,100.0,39966.0,COMPRESSION_READY,True
1,2,100.0,100.0,100.0,39965.0,COMPRESSION_READY,True
2,3,100.0,100.0,100.0,39965.0,COMPRESSION_READY,True


token_delta: -1.0

[POPULATION TRANSITIONS]


,generation,compression_ready,compression_phase,new_by_compression,new_by_micro_compression,new_by_block_compression,new_by_multi_block_compression,new_by_global_budget_compression,new_by_compression_fallback,new_by_advisor
0,1,True,COMPRESSION_READY,2,2,0,0,0,2,0
1,2,True,COMPRESSION_READY,2,2,0,0,0,2,0
2,3,True,COMPRESSION_READY,2,2,0,0,0,2,0



[ADVISOR PROMPT KEYWORDS]
advisor_prompt_generation_001.txt chars 53
  Advisor Case A                         False
  Advisor Case B                         False
  Advisor Case C                         False
  prompt_token_breakdown                 False
  block_token_breakdown                  False
  block_compression_proposals            False
  multi_block_compression_proposals      False
  global_budget_compression_proposals    False
advisor_prompt_generation_002.txt chars 53
  Advisor Case A                         False
  Advisor Case B                         False
  Advisor Case C                         False
  prompt_token_breakdown                 False
  block_token_breakdown                  False
  block_compression_proposals            False
  multi_block_compression_proposals      False
  global_budget_compression_proposals    False
advisor_prompt_generation_003.txt chars 53
  Advisor Case A                         False
  Advisor Case B                         Fals

,generation,proposal_id,schema_source,operator,compression_level,selected_block_id,selected_block_ids,expected_token_delta,accepted,rejection_reason,raw_response_path,advisor_prompt_path



[ADVISOR MUTATION PROPOSALS]
rows 0

[ALL COMPRESSION PROPOSALS]
rows 6


,_file,generation,source,schema_source,mutation_family,compression_level,operator,mutation_type,target_block_id,selected_block_id,selected_block_ids,expected_token_delta,measured_prompt_token_delta,accepted,rejection_reason,fallback_reason
0,mutation_proposals.jsonl,2,compression_fallback,,compression,micro,compress_candidate_strategies_to_minimal,compress_candidate_strategies_to_minimal,genome,,[],-2700,-1.0,True,,no_valid_advisor_proposal
1,mutation_proposals.jsonl,2,compression_fallback,,compression,micro,template_compress_rule_family,template_compress_rule_family,,,[],0,0.0,True,,no_valid_advisor_proposal
2,mutation_proposals.jsonl,3,compression_fallback,,compression,micro,lower_output_max_tokens_safe,lower_output_max_tokens_safe,genome,,[],-2698,0.0,True,,no_valid_advisor_proposal
3,mutation_proposals.jsonl,3,compression_fallback,,compression,micro,prune_micro_rules_to_top_k,prune_micro_rules_to_top_k,,,[],0,0.0,True,,no_valid_advisor_proposal
4,mutation_proposals.jsonl,4,compression_fallback,,compression,micro,reduce_few_shot_count_to_zero,reduce_few_shot_count_to_zero,genome,,[],-2698,NaN,True,,no_valid_advisor_proposal
5,mutation_proposals.jsonl,4,compression_fallback,,compression,micro,reduce_few_shot_count_to_zero,reduce_few_shot_count_to_zero,,,[],0,NaN,True,,no_valid_advisor_proposal


In [10]:
advisor_retry_out = run_ga_all_categories(use_advisor=True, base_prefix=f"smoke_advisor_strong_{SERVER_PRESET}")
inspect_run(advisor_retry_out)


RUN: qwen25_coder_14b / cloud_advisor
OUTPUT: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/smoke_advisor_strong_A6000_SET_A_cloud_advisor_cat1_lpc1_pop4_gens3_qwen25_coder_14b_20260611_201620/ga_output
PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10
MODEL_BASE: /home/mgjeong/Desktop/llm/local_models
MODEL_NAME: /home/mgjeong/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-14B-Instruct/snapshots/aedcc2d42b622764e023cf882b6652e646b95671
DEBUG_LOG: /tmp/joi_v15_worker_debug_qwen25_coder_14b_cloud_advisor_20260611_201620.log
COMMAND:
/home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10 -u /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --model-key qwen25_coder_14b --target-detpass 90 --llm-mode worker --population 4 --gens 3 --min-generations 3 --max-generations 3 --sample-size 1 --validation-size 1 --cheap-eval-limit 2 --candidate-k 1 --repair-attempts 0 --d

,generation,validation_det_pass_rate,validation_avg_det_score,best_so_far_DETPass,avg_prompt_tokens,compression_phase,compression_ready
0,1,100.0,100.0,100.0,39966.0,COMPRESSION_READY,True
1,2,100.0,100.0,100.0,39965.0,COMPRESSION_READY,True
2,3,100.0,100.0,100.0,39965.0,COMPRESSION_READY,True


token_delta: -1.0

[POPULATION TRANSITIONS]


,generation,compression_ready,compression_phase,new_by_compression,new_by_micro_compression,new_by_block_compression,new_by_multi_block_compression,new_by_global_budget_compression,new_by_compression_fallback,new_by_advisor
0,1,True,COMPRESSION_READY,2,2,0,0,0,2,0
1,2,True,COMPRESSION_READY,2,2,0,0,0,2,0
2,3,True,COMPRESSION_READY,2,2,0,0,0,2,0



[ADVISOR PROMPT KEYWORDS]
advisor_prompt_generation_001.txt chars 22114
  Advisor Case A                         False
  Advisor Case B                         True
  Advisor Case C                         False
  prompt_token_breakdown                 True
  block_token_breakdown                  True
  block_compression_proposals            True
  multi_block_compression_proposals      True
  global_budget_compression_proposals    True
advisor_prompt_generation_002.txt chars 19434
  Advisor Case A                         False
  Advisor Case B                         True
  Advisor Case C                         False
  prompt_token_breakdown                 True
  block_token_breakdown                  True
  block_compression_proposals            True
  multi_block_compression_proposals      True
  global_budget_compression_proposals    True
advisor_prompt_generation_003.txt chars 17790
  Advisor Case A                         False
  Advisor Case B                         True
  

,generation,proposal_id,schema_source,operator,compression_level,selected_block_id,selected_block_ids,expected_token_delta,accepted,rejection_reason,raw_response_path,advisor_prompt_path
0,1,advisor_g001_no_schema_diagnostic,no_schema,NaN,NaN,NaN,[],0,False,no_advisor_proposals_parsed,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...
1,2,advisor_g002_no_schema_diagnostic,no_schema,NaN,NaN,NaN,[],0,False,no_advisor_proposals_parsed,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...
2,3,advisor_g003_no_schema_diagnostic,no_schema,NaN,NaN,NaN,[],0,False,no_advisor_proposals_parsed,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...



[ADVISOR MUTATION PROPOSALS]
rows 3


,generation,source,proposal_id,schema_source,mutation_family,compression_level,operator,mutation_type,target_block_id,selected_block_id,selected_block_ids,target_block_family,expected_token_delta,measured_prompt_token_delta,accepted,rejection_reason,fallback_reason
0,1,advisor,advisor_g001_no_schema_diagnostic,no_schema,advisor_guided,,,,,,[],,0,None,False,no_advisor_proposals_parsed,
1,2,advisor,advisor_g002_no_schema_diagnostic,no_schema,advisor_guided,,,,,,[],,0,None,False,no_advisor_proposals_parsed,
2,3,advisor,advisor_g003_no_schema_diagnostic,no_schema,advisor_guided,,,,,,[],,0,None,False,no_advisor_proposals_parsed,



[ALL COMPRESSION PROPOSALS]
rows 6


,_file,generation,source,schema_source,mutation_family,compression_level,operator,mutation_type,target_block_id,selected_block_id,selected_block_ids,expected_token_delta,measured_prompt_token_delta,accepted,rejection_reason,fallback_reason
0,mutation_proposals.jsonl,2,compression_fallback,,compression,micro,compress_candidate_strategies_to_minimal,compress_candidate_strategies_to_minimal,genome,,[],-2700,-1.0,True,,no_valid_advisor_proposal
1,mutation_proposals.jsonl,2,compression_fallback,,compression,micro,template_compress_rule_family,template_compress_rule_family,,,[],0,0.0,True,,no_valid_advisor_proposal
2,mutation_proposals.jsonl,3,compression_fallback,,compression,micro,lower_output_max_tokens_safe,lower_output_max_tokens_safe,genome,,[],-2698,0.0,True,,no_valid_advisor_proposal
3,mutation_proposals.jsonl,3,compression_fallback,,compression,micro,prune_micro_rules_to_top_k,prune_micro_rules_to_top_k,,,[],0,0.0,True,,no_valid_advisor_proposal
4,mutation_proposals.jsonl,4,compression_fallback,,compression,micro,lower_output_max_tokens_safe,lower_output_max_tokens_safe,genome,,[],-2698,NaN,True,,no_valid_advisor_proposal
5,mutation_proposals.jsonl,4,compression_fallback,,compression,micro,reduce_few_shot_count_to_zero,reduce_few_shot_count_to_zero,,,[],0,NaN,True,,no_valid_advisor_proposal


In [11]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

OUT_DIR = Path(advisor_retry_out)

print("=" * 120)
print("[CHECK TRANSITIONS]")
print("=" * 120)

trans = pd.read_csv(OUT_DIR / "population_transitions.csv")
display(trans)

for col in [
    "new_by_micro_compression",
    "new_by_block_compression",
    "new_by_multi_block_compression",
    "new_by_global_budget_compression",
    "new_by_compression_fallback",
    "new_by_advisor",
]:
    if col in trans.columns:
        print(col, "sum =", pd.to_numeric(trans[col], errors="coerce").fillna(0).sum())

print("\n" + "=" * 120)
print("[CHECK BLOCK BREAKDOWN]")
print("=" * 120)

def load_json(path):
    path = Path(path)
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8", errors="replace"))

def extract_blocks(obj):
    if obj is None:
        return []
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for key in ["blocks", "block_token_breakdown", "block_breakdown", "items"]:
            if isinstance(obj.get(key), list):
                return obj[key]
        # dict 자체가 block_id -> block_info 형태일 수도 있음
        blocks = []
        for k, v in obj.items():
            if isinstance(v, dict):
                vv = dict(v)
                vv.setdefault("block_id", k)
                blocks.append(vv)
        return blocks
    return []

block_obj = load_json(OUT_DIR / "block_token_breakdown.json")
blocks = extract_blocks(block_obj)

print("block count:", len(blocks))

if blocks:
    bdf = pd.DataFrame(blocks)
    display(bdf)

    for col in ["token_estimate", "char_count", "few_shot_count", "micro_rule_count"]:
        if col in bdf.columns:
            bdf[col] = pd.to_numeric(bdf[col], errors="coerce")

    protected_col = "is_protected_block" if "is_protected_block" in bdf.columns else None
    allowed_col = "compression_allowed" if "compression_allowed" in bdf.columns else None

    if allowed_col:
        candidates = bdf[bdf[allowed_col].astype(bool)]
        if protected_col:
            candidates = candidates[~candidates[protected_col].astype(bool)]

        if "token_estimate" in candidates.columns:
            candidates = candidates.sort_values("token_estimate", ascending=False)

        print("\n[compressible non-protected block candidates]")
        display(candidates)

        print("candidate count:", len(candidates))
    else:
        print("[WARN] compression_allowed column not found.")
else:
    print("[WARN] No block breakdown blocks found.")

print("\n" + "=" * 120)
print("[CHECK ADVISOR PROPOSALS]")
print("=" * 120)

rows = []
p = OUT_DIR / "advisor_mutation_proposals.jsonl"
if p.exists():
    for line in p.read_text(encoding="utf-8", errors="replace").splitlines():
        if line.strip():
            try:
                rows.append(json.loads(line))
            except Exception:
                pass

print("advisor proposal rows:", len(rows))
if rows:
    df = pd.DataFrame(rows)
    cols = [
        "generation",
        "source",
        "proposal_id",
        "schema_source",
        "mutation_family",
        "compression_level",
        "operator",
        "target_block_id",
        "selected_block_id",
        "selected_block_ids",
        "target_block_family",
        "expected_token_delta",
        "accepted",
        "rejection_reason",
        "fallback_reason",
    ]
    cols = [c for c in cols if c in df.columns]
    display(df[cols])

[CHECK TRANSITIONS]


,generation,generation_phase,next_action,compression_ready,compression_phase,micro_compression_child_quota,block_compression_child_quota,multi_block_compression_child_quota,global_budget_compression_child_quota,compression_child_quota,...,duplicates_removed,duplicates_removed_by_prompt_hash,refill_reason,next_population,promotion_rejected,advisor_proposals_generated,advisor_proposals_accepted_applied,advisor_proposals_accepted_not_scheduled,advisor_proposals_rejected,advisor_children_scheduled
0,1,ACCURACY_SEARCH,continue_accuracy,True,COMPRESSION_READY,1,1,0,0,2,...,0,0,NaN,4,0,1,0,0,1,0
1,2,ACCURACY_SEARCH,continue_accuracy,True,COMPRESSION_READY,1,1,0,0,2,...,0,0,NaN,4,0,1,0,0,1,0
2,3,FINAL_SELECTION,stop_and_finalize,True,COMPRESSION_READY,1,1,0,0,2,...,1,0,dedupe_refill,4,0,1,0,0,1,0


new_by_micro_compression sum = 6
new_by_block_compression sum = 0
new_by_multi_block_compression sum = 0
new_by_global_budget_compression sum = 0
new_by_compression_fallback sum = 6
new_by_advisor sum = 0

[CHECK BLOCK BREAKDOWN]
block count: 0
[WARN] No block breakdown blocks found.

[CHECK ADVISOR PROPOSALS]
advisor proposal rows: 3


,generation,source,proposal_id,schema_source,mutation_family,compression_level,operator,target_block_id,selected_block_id,selected_block_ids,target_block_family,expected_token_delta,accepted,rejection_reason,fallback_reason
0,1,advisor,advisor_g001_no_schema_diagnostic,no_schema,advisor_guided,,,,,[],,0,False,no_advisor_proposals_parsed,
1,2,advisor,advisor_g002_no_schema_diagnostic,no_schema,advisor_guided,,,,,[],,0,False,no_advisor_proposals_parsed,
2,3,advisor,advisor_g003_no_schema_diagnostic,no_schema,advisor_guided,,,,,[],,0,False,no_advisor_proposals_parsed,


In [12]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

OUT_DIR = Path(advisor_retry_out)

print("=" * 120)
print("[A6000 BLOCK COMPRESSION DEBUG]")
print("=" * 120)
print("OUT_DIR:", OUT_DIR)

progress = pd.read_csv(OUT_DIR / "ga_generation_progress.csv")
trans = pd.read_csv(OUT_DIR / "population_transitions.csv")

print("\n[PROGRESS]")
cols = [
    "generation",
    "validation_det_pass_rate",
    "validation_avg_det_score",
    "best_so_far_DETPass",
    "avg_prompt_tokens",
    "compression_ready",
    "compression_phase",
    "generation_phase",
    "next_action",
]
display(progress[[c for c in cols if c in progress.columns]])

print("\n[TRANSITIONS]")
cols = [
    "generation",
    "compression_ready",
    "compression_phase",
    "micro_compression_child_quota",
    "block_compression_child_quota",
    "multi_block_compression_child_quota",
    "compression_child_quota",
    "new_by_micro_compression",
    "new_by_block_compression",
    "new_by_multi_block_compression",
    "new_by_compression",
    "new_by_compression_fallback",
    "new_by_advisor",
    "advisor_proposals_generated",
    "advisor_proposals_accepted_applied",
    "advisor_proposals_rejected",
]
display(trans[[c for c in cols if c in trans.columns]])

print("\n[SUMS]")
for col in [
    "new_by_micro_compression",
    "new_by_block_compression",
    "new_by_multi_block_compression",
    "new_by_compression",
    "new_by_compression_fallback",
    "new_by_advisor",
]:
    if col in trans.columns:
        print(col, "=", pd.to_numeric(trans[col], errors="coerce").fillna(0).sum())
    else:
        print(col, "= MISSING")

def read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8", errors="replace"))

def read_jsonl(path):
    rows = []
    path = Path(path)
    if not path.exists():
        return rows
    for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except Exception:
            pass
    return rows

def extract_blocks(obj):
    if obj is None:
        return []
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for k in ["blocks", "block_token_breakdown", "block_breakdown", "items", "rows", "block_rows"]:
            if isinstance(obj.get(k), list):
                return obj[k]
        rows = []
        for k, v in obj.items():
            if isinstance(v, dict):
                vv = dict(v)
                vv.setdefault("block_id", k)
                rows.append(vv)
        return rows
    return []

blocks = extract_blocks(read_json(OUT_DIR / "block_token_breakdown.json"))

compressible = []
for b in blocks:
    if not isinstance(b, dict):
        continue
    allowed = bool(b.get("compression_allowed", False))
    protected = bool(b.get("is_protected_block", False))
    tok = b.get("token_estimate", b.get("tokens", b.get("char_count", 0)))
    try:
        tok = float(tok or 0)
    except Exception:
        tok = 0
    if allowed and not protected and tok >= 50:
        compressible.append(b)

print("\n[BLOCK BREAKDOWN]")
print("block count:", len(blocks))
print("compressible non-protected block count:", len(compressible))
if blocks:
    display(pd.DataFrame(blocks))

print("\n[COMPRESSION PROPOSALS]")
rows = []
for fn in ["mutation_proposals.jsonl", "advisor_mutation_proposals.jsonl"]:
    for r in read_jsonl(OUT_DIR / fn):
        op = str(r.get("operator", r.get("mutation_type", "")))
        if r.get("mutation_family") == "compression" or "compress" in op or "few_shot" in op or "block" in op:
            r["_file"] = fn
            rows.append(r)

print("compression proposal rows:", len(rows))
if rows:
    df = pd.DataFrame(rows)
    cols = [
        "_file",
        "generation",
        "source",
        "schema_source",
        "mutation_family",
        "compression_level",
        "operator",
        "mutation_type",
        "target_block_id",
        "selected_block_id",
        "selected_block_ids",
        "expected_token_delta",
        "measured_prompt_token_delta",
        "accepted",
        "rejection_reason",
        "fallback_reason",
    ]
    display(df[[c for c in cols if c in df.columns]])

[A6000 BLOCK COMPRESSION DEBUG]
OUT_DIR: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/smoke_advisor_strong_A6000_SET_A_cloud_advisor_cat1_lpc1_pop4_gens3_qwen25_coder_14b_20260611_201620/ga_output

[PROGRESS]


,generation,validation_det_pass_rate,validation_avg_det_score,best_so_far_DETPass,avg_prompt_tokens,compression_ready,compression_phase,generation_phase,next_action
0,1,100.0,100.0,100.0,39966.0,True,COMPRESSION_READY,ACCURACY_SEARCH,continue_accuracy
1,2,100.0,100.0,100.0,39965.0,True,COMPRESSION_READY,ACCURACY_SEARCH,continue_accuracy
2,3,100.0,100.0,100.0,39965.0,True,COMPRESSION_READY,FINAL_SELECTION,stop_and_finalize



[TRANSITIONS]


,generation,compression_ready,compression_phase,micro_compression_child_quota,block_compression_child_quota,multi_block_compression_child_quota,compression_child_quota,new_by_micro_compression,new_by_block_compression,new_by_multi_block_compression,new_by_compression,new_by_compression_fallback,new_by_advisor,advisor_proposals_generated,advisor_proposals_accepted_applied,advisor_proposals_rejected
0,1,True,COMPRESSION_READY,1,1,0,2,2,0,0,2,2,0,1,0,1
1,2,True,COMPRESSION_READY,1,1,0,2,2,0,0,2,2,0,1,0,1
2,3,True,COMPRESSION_READY,1,1,0,2,2,0,0,2,2,0,1,0,1



[SUMS]
new_by_micro_compression = 6
new_by_block_compression = 0
new_by_multi_block_compression = 0
new_by_compression = 6
new_by_compression_fallback = 6
new_by_advisor = 0

[BLOCK BREAKDOWN]
block count: 3
compressible non-protected block count: 0


,generation,model_key,block_id,block_family,block_role,is_core_block,is_protected_block,char_count,token_estimate,few_shot_count,micro_rule_count,candidate_strategy_count,optional_status,current_params,compression_allowed,safe_mutation_types,measurement_method
0,3,qwen25_coder_14b,01,Core_System,core,True,True,14148,3537,0,0,0,,{},False,[],char_div_4_estimate
1,3,qwen25_coder_14b,02,Service_Mapping,core,True,True,27529,6883,2,2,0,,"{'few_shot_count': 2, 'micro_rules': ['Return ...",False,[],char_div_4_estimate
2,3,qwen25_coder_14b,03,Output_Schema,optional,False,True,1343,336,0,0,0,active,{},False,[],char_div_4_estimate



[COMPRESSION PROPOSALS]
compression proposal rows: 6


,_file,generation,source,schema_source,mutation_family,compression_level,operator,mutation_type,target_block_id,selected_block_id,selected_block_ids,expected_token_delta,measured_prompt_token_delta,accepted,rejection_reason,fallback_reason
0,mutation_proposals.jsonl,2,compression_fallback,,compression,micro,compress_candidate_strategies_to_minimal,compress_candidate_strategies_to_minimal,genome,,[],-2700,-1.0,True,,no_valid_advisor_proposal
1,mutation_proposals.jsonl,2,compression_fallback,,compression,micro,template_compress_rule_family,template_compress_rule_family,,,[],0,0.0,True,,no_valid_advisor_proposal
2,mutation_proposals.jsonl,3,compression_fallback,,compression,micro,lower_output_max_tokens_safe,lower_output_max_tokens_safe,genome,,[],-2698,0.0,True,,no_valid_advisor_proposal
3,mutation_proposals.jsonl,3,compression_fallback,,compression,micro,prune_micro_rules_to_top_k,prune_micro_rules_to_top_k,,,[],0,0.0,True,,no_valid_advisor_proposal
4,mutation_proposals.jsonl,4,compression_fallback,,compression,micro,lower_output_max_tokens_safe,lower_output_max_tokens_safe,genome,,[],-2698,NaN,True,,no_valid_advisor_proposal
5,mutation_proposals.jsonl,4,compression_fallback,,compression,micro,reduce_few_shot_count_to_zero,reduce_few_shot_count_to_zero,,,[],0,NaN,True,,no_valid_advisor_proposal


In [14]:
OUT_DIR=Path(advisor_retry_out); errors=[]
for name in ["block_token_breakdown.json","prompt_token_breakdown.json","ga_generation_progress.csv","population_transitions.csv","advisor_mutation_summary.csv","advisor_mutation_proposals.jsonl"]:
    if not (OUT_DIR/name).exists(): errors.append(f"missing artifact: {name}")
resp=sorted(OUT_DIR.glob("advisor_response_generation_*.json"))
if not resp: errors.append("missing advisor_response_generation_*.json")
else:
    ok=False
    for p in resp:
        obj=read_json(p); parsed=obj.get("parsed",{}) if isinstance(obj,dict) else {}
        if isinstance(parsed,dict) and "advisor_status" in parsed and "compression_policy" in parsed and any(k in parsed for k in ["proposals","mutation_proposals","micro_compression_proposals","block_compression_proposals","multi_block_compression_proposals","global_budget_compression_proposals"]): ok=True
    if not ok: errors.append("advisor parsed response does not preserve full proposal-bearing object")
if len(read_jsonl(OUT_DIR/"advisor_mutation_proposals.jsonl"))==0: errors.append("advisor_mutation_proposals.jsonl has zero rows")
if (OUT_DIR/"advisor_mutation_summary.csv").exists() and len(pd.read_csv(OUT_DIR/"advisor_mutation_summary.csv"))==0: errors.append("advisor_mutation_summary.csv is header-only")
progress=pd.read_csv(OUT_DIR/"ga_generation_progress.csv")
if "avg_prompt_tokens" not in progress.columns or not bool(pd.to_numeric(progress.get("avg_prompt_tokens"), errors="coerce").fillna(0).max()>0): errors.append("avg_prompt_tokens is not positive")
if "validation_det_pass_rate" in progress.columns and pd.to_numeric(progress["validation_det_pass_rate"], errors="coerce").fillna(0).max()<90: errors.append("DETPass below 90")
trans=pd.read_csv(OUT_DIR/"population_transitions.csv")
for col in ["new_by_micro_compression","new_by_block_compression"]:
    if col not in trans.columns: errors.append(f"population_transitions.csv missing {col}")
    elif pd.to_numeric(trans[col], errors="coerce").fillna(0).sum()<=0: errors.append(f"{col} did not schedule any child")
print("="*120); print("[FINAL GATE]"); print("="*120)
if errors:
    print("[DO NOT RUN FULL LOOP]")
    for e in errors: print("-", e)
    raise RuntimeError("Advisor staged compression smoke failed final gate.")
print("[OK] Advisor staged compression smoke passed final gate.")

print("READY_FOR_A6000_FULL_LOOP = True")
print("advisor_retry_out =", advisor_retry_out)

[FINAL GATE]
[DO NOT RUN FULL LOOP]
- new_by_block_compression did not schedule any child


RuntimeError: Advisor staged compression smoke failed final gate.

In [ ]:
RUN_COMPLEX_CATEGORY_SMOKE = False
if RUN_COMPLEX_CATEGORY_SMOKE:
    complex_out = run_ga_all_categories(use_advisor=True, categories=(3,4,5,6), sample_size=4, validation_size=4, population=5, gens=5, timeout_sec=2400, base_prefix=f"smoke_complex_strong_{SERVER_PRESET}")
    inspect_run(complex_out)
else:
    print("[SKIP] RUN_COMPLEX_CATEGORY_SMOKE=False")

In [ ]:
import matplotlib.pyplot as plt

def plot_token_curve(out_dir, title_prefix=""):
    df=pd.read_csv(Path(out_dir)/"ga_generation_progress.csv")
    if "generation" not in df.columns: df["generation"]=np.arange(1,len(df)+1)
    if "avg_prompt_tokens" in df.columns:
        plt.figure(figsize=(7,4)); plt.plot(df["generation"], pd.to_numeric(df["avg_prompt_tokens"], errors="coerce"), marker="o")
        plt.xlabel("Generation"); plt.ylabel("Average prompt tokens"); plt.title(f"{title_prefix} Prompt Token Trend".strip()); plt.grid(True); plt.show()
    if "validation_det_pass_rate" in df.columns:
        plt.figure(figsize=(7,4)); plt.plot(df["generation"], pd.to_numeric(df["validation_det_pass_rate"], errors="coerce"), marker="o")
        plt.xlabel("Generation"); plt.ylabel("DETPass (%)"); plt.title(f"{title_prefix} DETPass Trend".strip()); plt.grid(True); plt.show()

def plot_lane_counts(out_dir, title_prefix=""):
    df=pd.read_csv(Path(out_dir)/"population_transitions.csv")
    if "generation" not in df.columns: df["generation"]=np.arange(1,len(df)+1)
    cols=["new_by_micro_compression","new_by_block_compression","new_by_multi_block_compression","new_by_global_budget_compression","new_by_compression_fallback","new_by_advisor"]
    cols=[c for c in cols if c in df.columns]
    plt.figure(figsize=(8,4))
    for c in cols: plt.plot(df["generation"], pd.to_numeric(df[c], errors="coerce").fillna(0), marker="o", label=c)
    plt.xlabel("Generation"); plt.ylabel("Children scheduled"); plt.title(f"{title_prefix} Compression Lane Counts".strip()); plt.grid(True); plt.legend(); plt.show()

plot_token_curve(advisor_retry_out, "A6000 Advisor Strong Compression")
plot_lane_counts(advisor_retry_out, "A6000 Advisor Strong Compression")

In [ ]:
# ============================================================
# FINAL FULL FAIR RUN ON A6000
# cloudless vs cloud_advisor with staged strong compression
# ============================================================

from pathlib import Path
import os
import json
import time
import traceback
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 0. Safety checks
# ------------------------------------------------------------

assert "run_ga_all_categories" in globals(), "run_ga_all_categories is not defined."
assert "SERVER_PRESET" in globals(), "SERVER_PRESET is not defined."

if not os.environ.get("OPENAI_API_KEY", "").strip():
    raise RuntimeError("OPENAI_API_KEY is not set. Cloud advisor full run requires it.")

print("SERVER_PRESET:", SERVER_PRESET)
print("REPO:", REPO)
print("PYTHON:", PYTHON)
print("LOCAL_MODEL_BASE:", LOCAL_MODEL_BASE)

# ------------------------------------------------------------
# 1. Models / modes
# ------------------------------------------------------------

MODEL_LIST = [
    ("7B", "qwen25_coder_7b"),
    ("8B", "llama31_8b"),
    ("14B", "qwen25_coder_14b"),
]

RUN_MODES = [
    ("cloudless", False),
    ("cloud_advisor", True),
]

# ------------------------------------------------------------
# 2. Full GA config
# ------------------------------------------------------------

COMMON_GA_CONFIG = {
    "categories": tuple(range(1, 9)),
    "limit_per_category": 3,
    "sample_size": 24,
    "validation_size": 24,
    "population": 5,
    "gens": 10,
    "target_detpass": 90,
    "full_run": True,
    "progress": "verbose",
    "timeout_sec": 1800,
    "retries": 0,
}

# ------------------------------------------------------------
# 3. Strong compression config
# ------------------------------------------------------------

COMMON_COMPRESSION_KWARGS = dict(
    compression_detpass_threshold=90,
    aggressive_compression_after_target=True,

    compression_child_quota=1,
    compression_child_ratio=0.25,

    advisor_compression_child_quota=1,
    advisor_prefer_compression_after_detpass=90,

    compression_token_reduction_target=0.15,
    compression_token_plateau_delta=1.0,
    allow_aggressive_compression=True,

    micro_compression_child_quota=1,
    micro_compression_child_ratio=0.25,

    block_compression_child_quota=1,
    block_compression_child_ratio=0.25,

    multi_block_compression_child_quota=1,
    multi_block_compression_child_ratio=0.25,

    global_budget_compression_child_quota=0,

    enable_block_token_breakdown=True,
    enable_multi_block_compression=True,

    # 논문 primary run에서는 False 권장.
    # renderer 연결이 확실해지면 별도 ablation으로 True 실행.
    enable_render_budget_compression=False,

    min_compression_token_delta=50,
)

# ------------------------------------------------------------
# 4. Summary helper
# ------------------------------------------------------------

def summarize_full_run(out_dir):
    out_dir = Path(out_dir)

    row = {
        "out_dir": str(out_dir),
        "summary_exists": (out_dir / "ga_summary.json").exists(),
        "progress_exists": (out_dir / "ga_generation_progress.csv").exists(),
        "transition_exists": (out_dir / "population_transitions.csv").exists(),
        "mutation_proposals_exists": (out_dir / "mutation_proposals.jsonl").exists(),
        "advisor_proposals_exists": (out_dir / "advisor_mutation_proposals.jsonl").exists(),
        "block_breakdown_exists": (out_dir / "block_token_breakdown.json").exists(),
        "prompt_breakdown_exists": (out_dir / "prompt_token_breakdown.json").exists(),
        "best_DETPass": np.nan,
        "best_so_far_DETPass": np.nan,
        "first_avg_prompt_tokens": np.nan,
        "last_avg_prompt_tokens": np.nan,
        "token_delta": np.nan,
        "token_reduction_ratio": np.nan,
        "new_by_micro_compression": 0,
        "new_by_block_compression": 0,
        "new_by_multi_block_compression": 0,
        "new_by_global_budget_compression": 0,
        "new_by_compression_fallback": 0,
        "advisor_proposal_rows": 0,
        "advisor_accepted_rows": 0,
        "advisor_rejected_rows": 0,
    }

    summary_path = out_dir / "ga_summary.json"
    if summary_path.exists():
        try:
            s = json.loads(summary_path.read_text(encoding="utf-8", errors="replace"))
            row["best_DETPass"] = s.get("best_DETPass", s.get("best_so_far_DETPass", np.nan))
            row["best_so_far_DETPass"] = s.get("best_so_far_DETPass", np.nan)
        except Exception:
            pass

    progress_path = out_dir / "ga_generation_progress.csv"
    if progress_path.exists():
        try:
            progress = pd.read_csv(progress_path)
            if len(progress) and "avg_prompt_tokens" in progress.columns:
                tokens = pd.to_numeric(progress["avg_prompt_tokens"], errors="coerce")
                row["first_avg_prompt_tokens"] = float(tokens.iloc[0])
                row["last_avg_prompt_tokens"] = float(tokens.iloc[-1])
                row["token_delta"] = row["last_avg_prompt_tokens"] - row["first_avg_prompt_tokens"]
                if row["first_avg_prompt_tokens"]:
                    row["token_reduction_ratio"] = (
                        row["first_avg_prompt_tokens"] - row["last_avg_prompt_tokens"]
                    ) / row["first_avg_prompt_tokens"]

            if len(progress) and "best_so_far_DETPass" in progress.columns:
                row["best_so_far_DETPass"] = float(
                    pd.to_numeric(progress["best_so_far_DETPass"], errors="coerce").max()
                )
        except Exception:
            pass

    trans_path = out_dir / "population_transitions.csv"
    if trans_path.exists():
        try:
            trans = pd.read_csv(trans_path)
            for col in [
                "new_by_micro_compression",
                "new_by_block_compression",
                "new_by_multi_block_compression",
                "new_by_global_budget_compression",
                "new_by_compression_fallback",
            ]:
                if col in trans.columns:
                    row[col] = int(pd.to_numeric(trans[col], errors="coerce").fillna(0).sum())
        except Exception:
            pass

    advisor_jsonl = out_dir / "advisor_mutation_proposals.jsonl"
    if advisor_jsonl.exists():
        rows = []
        for line in advisor_jsonl.read_text(encoding="utf-8", errors="replace").splitlines():
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                pass

        row["advisor_proposal_rows"] = len(rows)
        row["advisor_accepted_rows"] = sum(1 for r in rows if r.get("accepted") is True)
        row["advisor_rejected_rows"] = sum(1 for r in rows if r.get("accepted") is False)

    return row

# ------------------------------------------------------------
# 5. Full fair loop
# ------------------------------------------------------------

ga_runs_fair = {}
ga_summary_rows = []

for label, model_key in MODEL_LIST:
    for mode_name, use_advisor in RUN_MODES:
        run_key = f"{label}_{mode_name}"

        print("\n" + "#" * 120)
        print(f"START RUN: {run_key} / {model_key}")
        print("#" * 120)

        try:
            out_dir = run_ga_all_categories(
                model_key=model_key,
                categories=COMMON_GA_CONFIG["categories"],
                limit_per_category=COMMON_GA_CONFIG["limit_per_category"],
                sample_size=COMMON_GA_CONFIG["sample_size"],
                validation_size=COMMON_GA_CONFIG["validation_size"],
                population=COMMON_GA_CONFIG["population"],
                gens=COMMON_GA_CONFIG["gens"],
                target_detpass=COMMON_GA_CONFIG["target_detpass"],
                base_prefix=f"ga_{SERVER_PRESET}_compression",
                use_advisor=use_advisor,
                full_run=COMMON_GA_CONFIG["full_run"],
                progress=COMMON_GA_CONFIG["progress"],
                timeout_sec=COMMON_GA_CONFIG["timeout_sec"],
                retries=COMMON_GA_CONFIG["retries"],

                advisor_trigger_mode="always" if use_advisor else "off",
                advisor_min_population_for_child=4,
                advisor_force_child_quota=True if use_advisor else False,
                use_mock_advisor=False,

                **COMMON_COMPRESSION_KWARGS,
            )

            ga_runs_fair[run_key] = out_dir

            row = summarize_full_run(out_dir)
            row["run_key"] = run_key
            row["model_key"] = model_key
            row["mode"] = mode_name
            ga_summary_rows.append(row)

            print(f"[PASS] {run_key}: {out_dir}")
            print("[SUMMARY]", row)

        except Exception as e:
            ga_runs_fair[run_key] = None

            err_row = {
                "run_key": run_key,
                "model_key": model_key,
                "mode": mode_name,
                "out_dir": None,
                "error": repr(e),
            }
            ga_summary_rows.append(err_row)

            print(f"[FAIL] {run_key}: {type(e).__name__}: {e}")
            traceback.print_exc()

        time.sleep(5)

# ------------------------------------------------------------
# 6. Final report
# ------------------------------------------------------------

valid_ga_runs_fair = {
    k: v for k, v in ga_runs_fair.items()
    if v is not None
}

print("\n" + "=" * 120)
print("VALID RUNS")
print("=" * 120)
for k, v in valid_ga_runs_fair.items():
    print(k, "=>", v)

summary_df = pd.DataFrame(ga_summary_rows)
display(summary_df)

if len(valid_ga_runs_fair) != len(MODEL_LIST) * len(RUN_MODES):
    print("[WARN] Some runs failed.")
else:
    print("[OK] All full fair runs completed.")

valid_ga_runs_fair